<a href="https://colab.research.google.com/github/tom-howes/w_and_b_lab/blob/main/imdb_wandb_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 1 – Experiment Tracking with W&B
**Dataset:** IMDb Sentiment (HuggingFace `datasets`)  
**Model:** PyTorch LSTM binary classifier  
**W&B logging:** loss/accuracy per epoch + per-layer gradient norms per batch

In [1]:
!pip install wandb datasets torch -qq

In [2]:
import wandb
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from collections import Counter

In [3]:
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: tom-howes01 (tom-howes01-northeastern-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
# All hyper-parameters in one dict so W&B can track them
config = dict(
    project    = "Lab1-visualize-models",
    run_name   = "pytorch-lstm-imdb",
    epochs     = 5,
    batch_size = 64,
    lr         = 1e-3,
    embed_dim  = 128,
    hidden_dim = 256,
    num_layers = 2,
    dropout    = 0.3,
    max_vocab  = 20_000,
    max_len    = 256,
    seed       = 42,
)

torch.manual_seed(config["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [5]:
# Load IMDb and build vocabulary
print("Loading IMDb dataset …")
raw = load_dataset("imdb")  # 25k train / 25k test

def simple_tokenise(text: str) -> list:
    return text.lower().split()

print("Building vocabulary …")
counter = Counter()
for example in raw["train"]:
    counter.update(simple_tokenise(example["text"]))

vocab = {"<pad>": 0, "<unk>": 1}
for word, _ in counter.most_common(config["max_vocab"] - 2):
    vocab[word] = len(vocab)

print(f"Vocabulary size: {len(vocab)}")

def encode(text: str, max_len: int) -> list:
    tokens = simple_tokenise(text)[:max_len]
    ids    = [vocab.get(t, 1) for t in tokens]   # 1 = <unk>
    ids   += [0] * (max_len - len(ids))           # 0 = <pad>
    return ids

Loading IMDb dataset …


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Building vocabulary …
Vocabulary size: 20000


In [6]:
# PyTorch Dataset & DataLoaders
class IMDbDataset(Dataset):
    def __init__(self, split: str):
        self.data = raw[split]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item  = self.data[idx]
        ids   = encode(item["text"], config["max_len"])
        label = item["label"]
        return (
            torch.tensor(ids,   dtype=torch.long),
            torch.tensor(label, dtype=torch.float),
        )

train_loader = DataLoader(IMDbDataset("train"), batch_size=config["batch_size"], shuffle=True)
test_loader  = DataLoader(IMDbDataset("test"),  batch_size=config["batch_size"])

In [7]:
# LSTM Model
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim,
            num_layers  = num_layers,
            dropout     = dropout if num_layers > 1 else 0,
            batch_first = True,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        emb = self.dropout(self.embedding(x))        # (B, L, E)
        _, (hidden, _) = self.lstm(emb)              # hidden: (layers, B, H)
        out = self.fc(self.dropout(hidden[-1]))       # last layer hidden state
        return out.squeeze(1)                         # (B,)

model = SentimentLSTM(
    vocab_size  = len(vocab),
    embed_dim   = config["embed_dim"],
    hidden_dim  = config["hidden_dim"],
    num_layers  = config["num_layers"],
    dropout     = config["dropout"],
).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"])

In [8]:
# Training helpers
def accuracy(logits, labels):
    preds = (torch.sigmoid(logits) >= 0.5).long()
    return (preds == labels.long()).float().mean().item()

def log_gradient_norms(step: int):
    """Log the L2 norm of gradients for every parameter layer to W&B."""
    grad_data = {}
    for name, param in model.named_parameters():
        if param.grad is not None:
            grad_data[f"grad_norm/{name}"] = param.grad.norm(2).item()
    wandb.log(grad_data, step=step)

def run_epoch(loader, train: bool, step: int):
    model.train(train)
    total_loss, total_acc, n = 0.0, 0.0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for ids, labels in loader:
            ids, labels = ids.to(device), labels.to(device)
            logits = model(ids)
            loss   = criterion(logits, labels)
            if train:
                optimizer.zero_grad()
                loss.backward()
                log_gradient_norms(step)          # ← W&B gradient logging
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                step += 1
            total_loss += loss.item()
            total_acc  += accuracy(logits, labels)
            n          += 1
    return total_loss / n, total_acc / n, step

In [9]:
# W&B run + training loop
run = wandb.init(
    project = config["project"],
    name    = config["run_name"],
    config  = config,
)

# log parameter histograms + gradients every 100 steps
wandb.watch(model, log="all", log_freq=100)

global_step = 0
for epoch in range(1, config["epochs"] + 1):
    train_loss, train_acc, global_step = run_epoch(train_loader, train=True,  step=global_step)
    val_loss,   val_acc,   _           = run_epoch(test_loader,  train=False, step=global_step)

    wandb.log({
        "epoch"      : epoch,
        "train/loss" : train_loss,
        "train/acc"  : train_acc,
        "val/loss"   : val_loss,
        "val/acc"    : val_acc,
    }, step=global_step)

    print(
        f"Epoch {epoch}/{config['epochs']} | "
        f"Train loss {train_loss:.4f} acc {train_acc:.4f} | "
        f"Val   loss {val_loss:.4f} acc {val_acc:.4f}"
    )

# Final summary metric (mirrors original run.summary usage)
run.summary["best_val_acc"] = val_acc

run.finish()

Epoch 1/5 | Train loss 0.6930 acc 0.5009 | Val   loss 0.6923 acc 0.5117
Epoch 2/5 | Train loss 0.6901 acc 0.5243 | Val   loss 0.6929 acc 0.5005
Epoch 3/5 | Train loss 0.6972 acc 0.5081 | Val   loss 0.6929 acc 0.5097
Epoch 4/5 | Train loss 0.6872 acc 0.5337 | Val   loss 0.6899 acc 0.5186
Epoch 5/5 | Train loss 0.6530 acc 0.6132 | Val   loss 0.6143 acc 0.6798


epoch,▁▃▅▆█
grad_norm/embedding.weight,▂▂▂▂▂▂▂▂▂▃▆▂▃▅▁▁▁▁▁▁▆▆█▅▅▄▄▆▃▂▄▄▅▄▅▅▆▄▆▆
grad_norm/fc.bias,▃▄▂▂▁▁▁▃▁▁▂▂▁▆█▃▆▆▁▄▃▄▁▆▂▁▁▁▄▄▁▂▄▃▅▆▂▂▂▅
grad_norm/fc.weight,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▄▂▄▄█▃▄▃▃▂▁▂▂▂▁▂▁▂▃▂▂▂▂
grad_norm/lstm.bias_hh_l0,▁▁▁▁▂▁▁▁▂▂▁▁▁▁▁▂▄▁▂▁▁▁▁▁▂▁▃▁▁▂▁▃▁▂▁█▂▅▄▁
grad_norm/lstm.bias_hh_l1,▄▄▂▃▃▃▇▂▄▃▄▃▂▂▃▂▂▁▁▁▁▁▁▁▁▁▂▂▂▂▅▂▂▄▂▄▄▃█▆
grad_norm/lstm.bias_ih_l0,▁▂▃▂▃▄▂▁▁▁▃▁▃▁▁▁▁▁▁▁▁▂▄▄▄▃▂▄▂▂▂▃▂▂▂▄▃▅▂█
grad_norm/lstm.bias_ih_l1,▅▃▃▄▄▄▅▄▄▃▃▃▃▃▆▄▁▁▁▁▁▁▂▂▂▃▂▂▂▂▂▂▂▃▃▂▂▃▅█
grad_norm/lstm.weight_hh_l0,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▃█▃▂▂▂▂▃▂▂▂▁▃▂▃
grad_norm/lstm.weight_hh_l1,▁▂▁▁▁▂▁▁▁▁▁█▁▁▁▃▆▁▁▁▁▂▁▁▁▂▂▁▂▁▂▂▁▁▃▂▂▂▆▃
+6,...
